In [ ]:
import sys

!which {sys.executable}

In [ ]:
import sys
sys.path.insert(0, '..')

# Kounkel+2020 catalog

In [ ]:
from wakai.catalog import CatalogDownloader

#no proper motion and radial velocity for members
cd = CatalogDownloader(catalog_name="Kounkel2020", catalog_type="cluster")
tabs = cd.get_tables_from_vizier(row_limit=-1)

In [ ]:
df = tabs[0].to_pandas()
df_mem = tabs[2].to_pandas()
df_mem.head()

In [ ]:
from wakai.catalog import CatalogDownloader

#no proper motion and radial velocity for members
cd2 = CatalogDownloader(catalog_name="CantatGaudin2020", catalog_type="cluster")
tabs2 = cd2.get_tables_from_vizier(row_limit=-1)

In [ ]:
df_mem2 = tabs2[1].to_pandas()

In [ ]:
#replace name to be compatible with other catalogs' naming formats
df_mem2 = df_mem2.rename({"Cluster": "Name",
                        "Source": "Gaia",
                        "Proba": "Prob",
                        "RA_ICRS": "ra",
                        "DE_ICRS": "dec",
                        "pmRA": "pmra",
                        "pmDE": "pmdec",
                        "Plx": "parallax",
                        "RV": "radial_velocity"
                        }, axis=1)
df_mem2.head()

# crossmatch Kounkel+20 and Cantat-Gaudin+20 for Gaia parameters

In [ ]:
import pandas as pd

# df_mem_merged = pd.merge(df_mem, df_mem2, on='Gaia', how='left').dropna(subset=['ra','dec'])
df_mem_merged = pd.merge(df_mem, df_mem2, on='Gaia')
df_mem_merged.head()

In [ ]:
df_mem_merged.shape

In [ ]:
from wakai.catalog import get_tois

tois = get_tois(clobber=False)
tois.tail()

In [ ]:
tois.shape

In [ ]:
tois[tois.Comments.apply(lambda x: 'young' in str(x))]

# cross-match TOI with catalog

In [ ]:
import pandas as pd

if True:
    #from astroquery.xmatch
    matches = pd.read_csv('../data/TOIs_GaiaDR3_xmatch_position.csv')
if False:
    #from vizier query/exofop
    matches = pd.read_csv('../data/TOIs_with_Gaiaid.csv')
    matches['GaiaDR3'] = matches['GaiaDR3_vizier']
matches.head()

In [ ]:
matches.shape

In [ ]:
final = pd.merge(df_mem_merged, matches, left_on='Gaia', right_on='GaiaDR3')
final.head()

In [ ]:
final2 = pd.merge(final, df[['Group','OName','logAge']], on='Group')
final.head()

In [ ]:
final2.shape

In [ ]:
final2.sort_values(by='TOI', ascending=False).to_csv(f'../data/{cd.catalog_name}/TOIs_in_{cd.catalog_name}.csv', index=False)